In the script below, i will be extracting medium from the jcm data with no name .


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Complex solutions_JCM.docx to Complex solutions_JCM.docx


In this code below i am going to try to convert word to csv files and the packages i will be using from  python arw docx and pandas

In [ ]:
import pandas as pd
from docx import Document #this package is installed because i want to convert word to csv file
#loading of the word document
doc = Document("Complex solutions_JCM.docx")
#extract the first table
table = doc.tables[0]
#convert table to list
data = []
for row in table.rows:
  row_data = [cell.text.strip() for cell in row.cells]
  data.append(row_data)
#first row becomes columns names
df = pd.DataFrame(data[1:], columns=data[0])
df.to_csv("Complex_solutions_JCM.csv", index = False)
print(df)

                               Complex solutions  \
0                                 Hemin solution   
1                             Menadione solution   
2                      VFA solution (medium 133)   
3                    Salt solution (medium 1000)   
4   Wolfe's modified mineral elixir (medium 221)   
..                                           ...   
85                       Solution A (medium 101)   
86                       Solution B (medium 101)   
87                  HEPES solution (medium 1004)   
88                  SW-25 solution (medium 1011)   
89                Vitamin solution (medium 1004)   

                                          Ingredients  
0       \mono{Hemin} {50} {mg} \n\mono{NAOH} {1} {ml}  
1   \mono{menadione} {5}{mg} \n\mono{ethanol}{1} {ml}  
2   \mono{Acetic acid}{17.0}{ml}\n\mono{Propionic ...  
3   \mono{CaCl$_2$$\cdot$2H$_2$O}{0.20}{g}\n\mono{...  
4   \mono{Nitrilotriacetic acid}{1.5}{g}\n\mono{Mg...  
..                                     

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving JCM_Ingredients_100_lines.csv to JCM_Ingredients_100_lines.csv


In [ ]:
#code to collect the first 100 rows from a file
df = pd.read_csv("JCM_Ingredients_100_lines.csv")
df_subset = df.head(100)
df_subset.to_csv("df_subset.csv", index = False)

The code below is used to convert txt document to csv and also thius document i am converting are lists of ingredients that are undivable ie their sub ingredients are not availble in the data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Undividable_Ingredeint.txt to Undividable_Ingredeint.txt


In [ ]:
import pandas as pd
import re
rows = []
pattern = r"\\mono\{(.+?)\}\{(.+?)\}\{(.+?)\}"
with open("Undividable_Ingredeint.txt", "r", encoding="utf-8") as f:
  for line in f:
    match = re.search(pattern, line)
    if match:
      ingredient = match.group(1)
      amount = match.group(2)
      unit = match.group(3)
      rows.append([ingredient, amount, unit])
df = pd.DataFrame(rows, columns=["ingredient", "amount" , "unit"])
df.to_csv("Undividable_ingredient.csv", index = False)

In [ ]:
import pandas as pd
df = pd.read_csv("merged_output_file_EDIT.csv")
df_subset = df.head(100)
df_subset.to_csv("df_subset.csv", index=False)
print("csv transformation succesfully!")

csv transformation succesfully!


The codes below i will be doing series of cleaning of three main files . The one that contain the ingredients , the complex solutions and the undivable solutions before writting them in RDF forms

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Complex_solutions_JCM.csv to Complex_solutions_JCM.csv


In [ ]:
import pandas as pd
import re
def format_components(text):
  if pd.isna(text):
    return text
  text = re.sub(r'(\\mono\{)', r'\n\1', text)
  text = re.sub(r'(\\chu\{)', r'\n\1', text)
  text = re.sub(r'\n+','\n', text)
  text = text.strip()
  return text
df = pd.read_csv("merged_one_fivty_2.csv")
df['ingredientName'] = df['ingredientName'].apply(format_components)
df.to_csv("formatted_152_rows.csv" , index= False)
print("sucesfully done!")


sucesfully done!


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Undividable_clean_ingredient.csv to Undividable_clean_ingredient.csv


In [ ]:
import pandas as pd #this code was mainly used to get rid of duplicates ingredients
df = pd.read_csv("Undividable_clean_ingredient.csv")
df_unique = df.drop_duplicates(subset=["ingredient"], keep="first")
df_unique.to_csv("ingredients_unique.csv", index= False)

In [ ]:
from google.colab import files
imported_files = files.upload()

Saving Complex_solutions_JCM.csv to Complex_solutions_JCM (1).csv


In [ ]:
import pandas as pd
import re
df = pd.read_csv("Complex_solutions_JCM.csv")
clean_rows = []
pattern = re.compile(
    r"\\mono\s*\{(.*?)\}\s*\{(.*?)\}\s*\{(.*?)\}",
    re.DOTALL
)
for idx, row in df. iterrows():
  solution_name = row["Complex solutions"]
  ingredients_text = str(row["Ingredients"])
  matches = pattern.findall(ingredients_text)
  for ing, amt, unit in matches:
    ing = ing.replace("\n", " ").strip()
    amt = amt.strip()
    unit = unit.strip()
    clean_rows.append({
        "complex_solution":solution_name,
        "ingredient":ing,
        "amount":amt,
        "unit":unit
    })
clean_df = pd.DataFrame(clean_rows)
clean_df.to_csv("parsed_ingredient.csv", index = False)

In [ ]:
import re
import pandas as pd
def format_components(text):
  if pd.isna(text):
    return text
  text = re.sub(r'\\chu{.*?\}','', text) #this code remove the \chu and anything insdie the chu
  #text = re.sub(r'\\chu\s*[^}]*\}', '', text) #handling broken \chu
  def clean_mono(match):
    name = match.group(1)
    value = match.group(2)
    unit = match.group(3)
    name = re.sub(r'\(see Medium No\.\s*\[.*?\]\]\)', '', name)
    name = name.strip()
    return f"\\mono{{{name}}}{{{value}}}{{{unit}}}"
  text = re.sub(r'\\mono{(.*?)\}\s*\{(.*?)\}s*\{(.*?)\}', clean_mono,
                text
                )
  return text
df = pd.read_csv("formatted_152_rows_editt.csv")
df["cleaned"] = df["ingredientName"].apply(format_components)
df.to_csv("formatted_new_edit_1.csv", index = False)
print("converstion completed")

converstion completed


In the code below we are extracting medium components that are normal with no faults

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_RDF_ready_Final.csv to cleaned_RDF_ready_Final.csv


In [ ]:
import pandas as pd

# Load the cleaned file you generated earlier
df = pd.read_csv("cleaned_RDF_ready_Final.csv")

# Normalize ingredient names (optional but recommended)
df["ingredientName"] = df["ingredientName"].str.strip()

# Count occurrences
ingredient_counts = df["ingredientName"].value_counts().reset_index()

# Rename columns for clarity
ingredient_counts.columns = ["ingredientName", "count"]

# Save the result
ingredient_counts.to_csv("ingredient_counts.csv", index=False)

print("Ingredient counting completed!")
print(ingredient_counts.head(20))


Ingredient counting completed!
                                   ingredientName  count
0                                 Distilled water     89
1                                            NaCl     47
2                                          KH2PO4     37
3                                           NH4Cl     36
4                                             KCl     34
5                                      CaCl2.2H2O     34
6                                      MgCl2.6H2O     30
7                                       Resazurin     30
8                                   Yeast extract     29
9                                      MgSO4.7H2O     29
10                                         K2HPO4     28
11                       Yeast extract (BD-Difco)     24
12          Trace vitamins (see Medium No. [197])     23
13                                        Glucose     20
14                                           Agar     20
15                          5% Na2S.9H2O solution     18


In [ ]:
import pandas as pd
import re
def split_into_lines(text):
  if pd.isna(text):
    return text
  text = re.sub(r'\s*(\\mono|\\chu)', r'\n\1', text)
  return text.strip()
def remove_chu(text):
  if pd.isna(text):
    return text
  text = re.sub(r'\\chu\{.*?\}', '' , text , flags=re.DOTALL)
  return text.strip()
def normalise_mono(text):
  if pd.isna(text):
    return text
  text = re.sub(r'(?<!\\)mono', r'\\mono', text)
  text = re.sub(r'\\mono\s*\{([^}]*)\}\s*\{([^}]*)\}\s*\{([^}]*)\}',
                  r'\\mono{\1}{\2}{\3}', text)
  return text
def extract_mono_components(text):
  if pd.isna(text):
    return []
  pattern = r'\\mono\{(.*?)\}\{(.*?)\}\{(.*?)\}'
  matches = re.findall(pattern, text)
  ingredients = []
  for name, amount , unit in matches:
    ingredients.append({
        "ingredientName":name.strip(),
        "ingredientAmount":amount.strip() ,
        "ingredientUnit":unit.strip()
    })
df = pd.read_csv("df_subset_clean.csv", header=None, engine="python")
df["step1"] = df["ingredientName"].apply(split_into_lines)
df["step2"] = df["step1"].apply(remove_chu)
df["step3"] = df["step2"].apply(normalise_mono)
df["parsed"] = df["step3"].apply(extract_mono_components)
rows = []
for idx, row in df.iterrows():
  for ing in row["parsed"]:
    rows.append({
      "mediumID":row["mediumID"],
      "mediumName":row["mediumName"],
      "ingredientName":ing["ingredientName"],
      "ingredientAmount":ing["ingredientAmount"],
      "ingredientUnit":ing["ingredientUnit"]
    })
clean_df = pd.DataFrame(rows)
clean_df.to_csv("cleaned_RDF_ready.csv", index=False)
print("Cleaning succesfully!")

KeyError: 'ingredientName'

In [ ]:
import pandas as pd
import re

def split_into_lines(text):
    if pd.isna(text):
        return text
    text = re.sub(r'\s*(\\mono|\\chu)', r'\n\1', text)
    return text.strip()

def remove_chu(text):
    if pd.isna(text):
        return text
    text = re.sub(r'\\chu\{.*?\}', '' , text , flags=re.DOTALL)
    return text.strip()

def normalise_mono(text):
    if pd.isna(text):
        return text
    text = re.sub(r'(?<!\\)mono', r'\\mono', text)
    text = re.sub(r'\\mono\s*\{([^}]*)\}\s*\{([^}]*)\}\s*\{([^}]*)\}',
                  r'\\mono{\1}{\2}{\3}', text)
    return text

def extract_mono_components(text):
    if pd.isna(text):
        return []
    pattern = r'\\mono\{(.*?)\}\{(.*?)\}\{(.*?)\}'
    matches = re.findall(pattern, text)
    ingredients = []
    for name, amount , unit in matches:
        ingredients.append({
            "ingredientName":name.strip(),
            "ingredientAmount":amount.strip(),
            "ingredientUnit":unit.strip()
        })
    return ingredients

# ---------------------------------------------------------
# ⭐ FIXED LOADING SECTION
# ---------------------------------------------------------
df = pd.read_csv("df_subset_clean.csv", header=None, engine="python")
df.columns = ["mediumID", "mediumName", "ingredientName"]

# ---------------------------------------------------------
# Continue with your cleaning pipeline
# ---------------------------------------------------------
df["step1"] = df["ingredientName"].apply(split_into_lines)
df["step2"] = df["step1"].apply(remove_chu)
df["step3"] = df["step2"].apply(normalise_mono)
df["parsed"] = df["step3"].apply(extract_mono_components)

rows = []
for idx, row in df.iterrows():
    for ing in row["parsed"]:
        rows.append({
            "mediumID":row["mediumID"],
            "mediumName":row["mediumName"],
            "ingredientName":ing["ingredientName"],
            "ingredientAmount":ing["ingredientAmount"],
            "ingredientUnit":ing["ingredientUnit"]
        })

clean_df = pd.DataFrame(rows)
clean_df.to_csv("cleaned_RDF_ready.csv", index=False)
print("Cleaning successfully!")


Cleaning successfully!


In [ ]:
import pandas as pd

# Load the cleaned file you generated earlier
df = pd.read_csv("cleaned_RDF_ready.csv")

# Normalize ingredient names (optional but recommended)
df["ingredientName"] = df["ingredientName"].str.strip().str.lower()

# Count occurrences
ingredient_counts = df["ingredientName"].value_counts().reset_index()

# Rename columns for clarity
ingredient_counts.columns = ["ingredientName", "count"]

# Save the result
ingredient_counts.to_csv("ingredient_counts.csv", index=False)

print("Ingredient counting completed!")
print(ingredient_counts.head(20))


In [ ]:


# Load file without assuming tabs
df = pd.read_csv("df_subset_clean.csv", header=None, engine="python")

# Rename columns
df.columns = ["mediumID", "mediumName", "ingredientName"]

print(df.head())


   mediumID                 mediumName  \
0  mediumID                 mediumName   
1         1                 MRS MEDIUM   
2       100     ALKALINE NUTRIENT AGAR   
3      1000                PYVG MEDIUM   
4      1001  MARINE AGAR 2216 (pH 8.5)   

                                      ingredientName  
0                                     ingredientName  
1  \mono{Casein peptone, tryptic digest}{10.0}{g}...  
2  \mono{Peptone}{5.0}{g}\n\mono{Beef extract}{3....  
3  \mono{Trypticase peptone (BD-BBL)}{5.0}{g}\n\m...  
4  \mono{10% Na2CO3 solution.}\n\mono{Marine agar...  


In [ ]:
import re
import csv

# =========================
# FILE PATHS
# =========================
input_file = "jcm_components.txt"
output_file = "media_structured_JCM.csv"

# =========================
# READ RAW TEXT
# =========================
with open(input_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Normalize line endings
raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

# =========================
# FIND START OF EACH MEDIUM
# Medium starts with:
#   number + uppercase medium name + then \mono or \chu
# =========================
medium_start_pattern = re.compile(
    r'(?m)^(\d{1,4})\s+([A-Z0-9\-\(\)\[\]\'’&,/\. :]+?)\s+(?=\\mono|\\chu)'
)

matches = list(medium_start_pattern.finditer(raw_text))

# =========================
# HELPER: Extract one balanced {...} block from a given "{"
# Returns (full_block, end_index_after_block)
# Example:
#   text[start] must be "{"
#   returns ("{NaCl}", index_after_closing_brace)
# =========================
def extract_balanced_brace_block(text, start):
    if start >= len(text) or text[start] != "{":
        return None, start

    depth = 0
    j = start

    while j < len(text):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[start:j+1], j + 1
        j += 1

    # Malformed braces
    return None, start

# =========================
# FUNCTION: Extract all \mono{...}{...}{...} and \chu{...}
# in original order, preserving full \mono with value + unit
# =========================
def extract_ordered_blocks(text):
    """
    Extract all \\mono and \\chu blocks in the order they appear.

    For \\mono:
        capture full component line like:
        \\mono{NaCl}{37.8}{g}
        or
        \\mono{NaCl} {37.8} {g}

    For \\chu:
        capture only the full \\chu{...} block.

    Returns a list of strings.
    """
    blocks = []
    i = 0

    while i < len(text):
        mono_pos = text.find("\\mono{", i)
        chu_pos = text.find("\\chu{", i)

        # No more blocks
        if mono_pos == -1 and chu_pos == -1:
            break

        # Determine next block in order
        if mono_pos == -1:
            start = chu_pos
            command = "chu"
        elif chu_pos == -1:
            start = mono_pos
            command = "mono"
        else:
            if mono_pos < chu_pos:
                start = mono_pos
                command = "mono"
            else:
                start = chu_pos
                command = "chu"

        token = f"\\{command}"
        brace_start = start + len(token)

        # Must begin with {
        if brace_start >= len(text) or text[brace_start] != "{":
            i = start + 1
            continue

        # Extract first balanced block after \mono or \chu
        first_block, next_pos = extract_balanced_brace_block(text, brace_start)

        if first_block is None:
            i = start + 1
            continue

        if command == "chu":
            # \chu only has one main block
            full_block = token + first_block
            blocks.append(full_block.strip())
            i = next_pos

        elif command == "mono":
            # Start with ingredient block
            full_block = token + first_block
            pos = next_pos

            # Skip whitespace/newlines after ingredient
            while pos < len(text) and text[pos].isspace():
                pos += 1

            # Try to capture value block {37.8}
            if pos < len(text) and text[pos] == "{":
                second_block, pos2 = extract_balanced_brace_block(text, pos)
                if second_block is not None:
                    full_block += second_block
                    pos = pos2

                    # Skip whitespace/newlines after value
                    while pos < len(text) and text[pos].isspace():
                        pos += 1

                    # Try to capture unit block {g}
                    if pos < len(text) and text[pos] == "{":
                        third_block, pos3 = extract_balanced_brace_block(text, pos)
                        if third_block is not None:
                            full_block += third_block
                            pos = pos3

            blocks.append(full_block.strip())
            i = pos

    return blocks

# =========================
# PROCESS EACH MEDIUM
# =========================
media_rows = []

for idx, match in enumerate(matches):
    grmd = match.group(1).strip()
    md_name = match.group(2).strip()

    # Medium text begins after the medium name
    block_start = match.end()
    block_end = matches[idx + 1].start() if idx < len(matches) - 1 else len(raw_text)
    block_text = raw_text[block_start:block_end].strip()

    # Extract ALL \mono and \chu blocks in original order
    ordered_blocks = extract_ordered_blocks(block_text)

    # Put everything into ONE column
    media_text = "\n".join(ordered_blocks).strip()

    media_rows.append({
        "grmd": grmd,
        "md_name": md_name,
        "media_text": media_text
    })

# =========================
# WRITE OUTPUT CSV
# =========================
with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["grmd", "md_name", "media_text"])
    writer.writeheader()
    writer.writerows(media_rows)

print(f"Done! Created {output_file} with {len(media_rows)} media rows.")

Done! Created media_structured_JCM.csv with 975 media rows.


In the code below we re extracting medium and components that are difficult

In [ ]:
import re
import csv

# =========================
# FILE PATHS
# =========================
input_file = "jcm_components.txt"
output_file = "problematic_media_with_components.csv"

# =========================
# grmd VALUES TO REMOVE FROM FINAL OUTPUT
# =========================
remove_grmd = {"1", "5", "200", "10", "60", "18", "500", "1046", "120", "45", "50", "20"}

# =========================
# READ RAW TEXT
# =========================
with open(input_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Normalize line endings
raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

# =========================
# FIND MEDIUM BLOCKS
# Each medium block starts at a line beginning with digits
# and ends before the next such line
# =========================
medium_block_pattern = re.compile(
    r'(?ms)^(\d{1,4})\s+(.*?)(?=^\d{1,4}\s+|\Z)'
)

matches = list(medium_block_pattern.finditer(raw_text))

print(f"Found {len(matches)} candidate blocks.")

# =========================
# HELPER: Split medium name from body
# Everything before first \mono or \chu = md_name
# Everything from first \mono or \chu onward = components/body
# =========================
def split_name_and_body(rest):
    body_match = re.search(r'\\(?:mono|chu)\{', rest)
    if body_match:
        md_name = rest[:body_match.start()].strip()
        body = rest[body_match.start():].strip()
    else:
        md_name = rest.strip()
        body = ""
    return md_name, body

# =========================
# HELPER: Extract one balanced {...} block from a given "{"
# Returns (full_block, end_index_after_block)
# Example:
#   text[start] must be "{"
#   returns ("{NaCl}", index_after_closing_brace)
# =========================
def extract_balanced_brace_block(text, start):
    if start >= len(text) or text[start] != "{":
        return None, start

    depth = 0
    j = start

    while j < len(text):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[start:j+1], j + 1
        j += 1

    # Malformed braces
    return None, start

# =========================
# HELPER: Extract all \mono and \chu blocks in original order
# For \mono, keep ingredient + value + unit if present
# =========================
def extract_ordered_blocks(text):
    blocks = []
    i = 0

    while i < len(text):
        mono_pos = text.find("\\mono{", i)
        chu_pos = text.find("\\chu{", i)

        if mono_pos == -1 and chu_pos == -1:
            break

        # Find whichever comes first
        if mono_pos == -1:
            start = chu_pos
            command = "chu"
        elif chu_pos == -1:
            start = mono_pos
            command = "mono"
        else:
            if mono_pos < chu_pos:
                start = mono_pos
                command = "mono"
            else:
                start = chu_pos
                command = "chu"

        token = f"\\{command}"
        brace_start = start + len(token)

        # Safety check
        if brace_start >= len(text) or text[brace_start] != "{":
            i = start + 1
            continue

        # Extract first balanced block after \mono or \chu
        first_block, next_pos = extract_balanced_brace_block(text, brace_start)

        if first_block is None:
            i = start + 1
            continue

        if command == "chu":
            # \chu only has one main block
            full_block = token + first_block
            blocks.append(full_block.strip())
            i = next_pos

        elif command == "mono":
            # Start with ingredient block
            full_block = token + first_block
            pos = next_pos

            # Skip whitespace/newlines after ingredient
            while pos < len(text) and text[pos].isspace():
                pos += 1

            # Try to capture value block {37.8}
            if pos < len(text) and text[pos] == "{":
                second_block, pos2 = extract_balanced_brace_block(text, pos)
                if second_block is not None:
                    full_block += second_block
                    pos = pos2

                    # Skip whitespace/newlines after value
                    while pos < len(text) and text[pos].isspace():
                        pos += 1

                    # Try to capture unit block {g}
                    if pos < len(text) and text[pos] == "{":
                        third_block, pos3 = extract_balanced_brace_block(text, pos)
                        if third_block is not None:
                            full_block += third_block
                            pos = pos3

            blocks.append(full_block.strip())
            i = pos

    return blocks

# =========================
# HELPER: Mixed-case token detector
# Examples:
#   NaCl  -> True
#   YpSs  -> True
#   MEDIUM -> False
#   AGAR -> False
# =========================
def has_mixed_case_token(name):
    tokens = re.findall(r"[A-Za-z0-9\+\-/\*'%]+", name)

    for token in tokens:
        has_upper = any(c.isupper() for c in token)
        has_lower = any(c.islower() for c in token)

        if has_upper and has_lower:
            return True

    return False

# =========================
# HELPER: Lowercase whitelist token detector
# Example:
#   x in "2 x YT MEDIUM"
#   g in "200 g SUCROSE"
# =========================
def has_whitelisted_lowercase_token(name):
    whitelist = {"x", "g", "ml", "l"}
    tokens = re.findall(r"[A-Za-z]+", name)

    for token in tokens:
        if token in whitelist:
            return True

    return False

# =========================
# HELPER: Detect issue types
# Only the categories you want
# =========================
def detect_issues(md_name):
    issues = []
    name = md_name.strip()

    # 1. No medium name
    if name == "":
        issues.append("no_name")
        return issues

    # 2. Contains %
    if "%" in name:
        issues.append("contains_percent")

    # 3. Contains *
    if "*" in name:
        issues.append("contains_asterisk")

    # 4. Mixed-case token like NaCl, YpSs
    if has_mixed_case_token(name):
        issues.append("mixed_case_token")

    # 5. Lowercase whitelist token like x, g, ml, l
    if has_whitelisted_lowercase_token(name):
        issues.append("contains_whitelisted_lowercase_token")

    return issues

# =========================
# HELPER: Clean medium name
# Convert medium name to uppercase ONLY
# (do NOT change components)
# =========================
def clean_md_name(md_name):
    cleaned = md_name.upper().strip()
    cleaned = re.sub(r"\s+", " ", cleaned)  # collapse extra spaces
    return cleaned

# =========================
# PROCESS
# =========================
problem_rows = []

for m in matches:
    grmd = m.group(1).strip()
    rest = m.group(2).strip()

    # Keep only blocks that look like real media
    if "\\mono{" not in rest and "\\chu{" not in rest:
        continue

    md_name_original, body = split_name_and_body(rest)

    # Extract ordered components + method text together in ONE column
    ordered_blocks = extract_ordered_blocks(body)
    components_text = "\n".join(ordered_blocks).strip()

    # Detect issues
    issues = detect_issues(md_name_original)

    # Only keep problematic ones
    if not issues:
        continue

    # Clean medium name
    md_name_cleaned = clean_md_name(md_name_original)

    problem_rows.append({
        "grmd": grmd,
        "md_name_original": md_name_original,
        "md_name_cleaned": md_name_cleaned,
        "components_text": components_text,
        "issue_types": "; ".join(issues)
    })

# =========================
# REMOVE SPECIFIC BAD ROWS FROM FINAL RESULT
# =========================
original_count = len(problem_rows)

problem_rows = [row for row in problem_rows if row["grmd"] not in remove_grmd]

removed_count = original_count - len(problem_rows)

# =========================
# WRITE OUTPUT CSV
# =========================
with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "grmd",
            "md_name_original",
            "md_name_cleaned",
            "components_text",
            "issue_types"
        ]
    )
    writer.writeheader()
    writer.writerows(problem_rows)

print(f"Done! Created {output_file} with {len(problem_rows)} rows.")
print(f"Removed {removed_count} specified bad rows from final output.")

# =========================
# OPTIONAL SUMMARY
# =========================
summary = {
    "no_name": 0,
    "contains_percent": 0,
    "contains_asterisk": 0,
    "mixed_case_token": 0,
    "contains_whitelisted_lowercase_token": 0
}

for row in problem_rows:
    for issue in row["issue_types"].split("; "):
        if issue in summary:
            summary[issue] += 1

print("\nIssue summary (AFTER removing specified grmd rows):")
for k, v in summary.items():
    print(f"{k}: {v}")

Found 1180 candidate blocks.
Done! Created problematic_media_with_components.csv with 127 rows.
Removed 18 specified bad rows from final output.

Issue summary (AFTER removing specified grmd rows):
no_name: 0
contains_percent: 82
contains_asterisk: 1
mixed_case_token: 89
contains_whitelisted_lowercase_token: 5


In the code below we are extracting medium and components with no name

In [ ]:
import re
import pandas as pd

# =========================
# 1. FILE PATHS
# =========================
input_file = "jcm_components.txt"
output_file = "jcm_cleaned_medium_data.csv"

# =========================
# 2. READ RAW TEXT
# =========================
with open(input_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Normalize line endings
raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

# =========================
# 3. FIND CANDIDATE MEDIUM BLOCKS
# Each block starts at line beginning with digits
# and ends before next line beginning with digits
# =========================
medium_block_pattern = re.compile(
    r'(?ms)^(\d{1,4})\s*(.*?)(?=^\d{1,4}\s+|\Z)'
)

matches = list(medium_block_pattern.finditer(raw_text))
print(f"Found {len(matches)} candidate blocks.")

# =========================
# 4. FUNCTION: SPLIT NAME AND BODY
# Everything before first \mono or \chu = possible md_name
# =========================
def split_name_and_body(rest):
    body_match = re.search(r'\\(?:mono|chu)\{', rest)

    if body_match:
        md_name = rest[:body_match.start()]
        body = rest[body_match.start():]
    else:
        md_name = rest
        body = ""

    return md_name, body

# =========================
# 5. FUNCTION: STRICT NO-NAME DETECTOR
# True if md_name is:
# - empty
# - whitespace only
# - only backslashes (e.g. "\" or "\\")
# =========================
def is_true_no_name(md_name):
    cleaned = md_name.strip()

    if cleaned == "":
        return True

    if re.fullmatch(r'\\+', cleaned):
        return True

    return False

# =========================
# 6. HELPER: EXTRACT ONE BALANCED {...} GROUP
# Returns (content_inside, position_after_group)
# =========================
def extract_brace_group(text, start_pos):
    if start_pos >= len(text) or text[start_pos] != "{":
        return None, start_pos

    depth = 0
    j = start_pos

    while j < len(text):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                content = text[start_pos + 1:j]
                return content.strip(), j + 1
        j += 1

    # malformed brace
    return None, start_pos

# =========================
# 7. HELPER: FORMAT BLOCK CONTENT
# Convert:
#   \mono{NaCl}{g}{5.0}      -> NaCl (5.0) (g)
#   \mono{NaCl}{5.0}{g}      -> NaCl (5.0) (g)
#   \mono{Peptone}           -> Peptone
#   \chu{Adjust pH to 7.0}   -> Adjust pH to 7.0
#
# We try to detect which extra brace is numeric (value)
# and which is unit.
# =========================
def format_block_content(command, first_content, extra_groups):
    first_content = re.sub(r"\s+", " ", first_content).strip()
    extras = [re.sub(r"\s+", " ", x).strip() for x in extra_groups if x.strip()]

    # For \chu, usually just combine naturally
    if command == "chu":
        if extras:
            return first_content + " " + " ".join(f"({x})" for x in extras)
        return first_content

    # For \mono, try to identify value and unit
    value = None
    unit = None
    other_extras = []

    for x in extras:
        # numeric-like: 5, 5.0, 0.25, -1.2
        if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", x):
            if value is None:
                value = x
            else:
                other_extras.append(x)
        else:
            # if contains digits + letters, it may still be unit like mg, g/L, %, etc.
            if unit is None:
                unit = x
            else:
                other_extras.append(x)

    # Build clean final text
    result = first_content

    if value is not None:
        result += f" ({value})"
    if unit is not None:
        result += f" ({unit})"

    # If anything else remains, keep it too (to avoid data loss)
    for x in other_extras:
        result += f" ({x})"

    return result.strip()

# =========================
# 8. FUNCTION: EXTRACT ALL \mono AND \chu BLOCKS
# INCLUDING EXTRA {...}{...} AFTER THE FIRST BLOCK
# =========================
def extract_all_blocks(body):
    blocks = []
    i = 0

    while i < len(body):
        mono_pos = body.find(r"\mono{", i)
        chu_pos = body.find(r"\chu{", i)

        if mono_pos == -1 and chu_pos == -1:
            break

        # Decide which comes first
        if mono_pos == -1:
            start = chu_pos
            command = "chu"
        elif chu_pos == -1:
            start = mono_pos
            command = "mono"
        else:
            if mono_pos < chu_pos:
                start = mono_pos
                command = "mono"
            else:
                start = chu_pos
                command = "chu"

        token = f"\\{command}"
        brace_start = start + len(token)

        # Must start with {
        if brace_start >= len(body) or body[brace_start] != "{":
            i = start + 1
            continue

        # 1) Extract first required brace group
        first_content, pos_after = extract_brace_group(body, brace_start)

        if first_content is None:
            # malformed, skip safely
            i = start + 1
            continue

        # 2) Extract immediately following extra {...} groups
        extra_groups = []
        pos = pos_after

        while pos < len(body):
            # skip whitespace between groups
            while pos < len(body) and body[pos].isspace():
                pos += 1

            if pos < len(body) and body[pos] == "{":
                extra_content, new_pos = extract_brace_group(body, pos)
                if extra_content is None:
                    break
                extra_groups.append(extra_content)
                pos = new_pos
            else:
                break

        # 3) Format the block nicely
        final_text = format_block_content(command, first_content, extra_groups)

        if final_text:
            blocks.append(final_text)

        # Move pointer after this full block + extras
        i = pos

    # Join all cleaned components into ONE ingredientName column
    return "\n".join(blocks)

# =========================
# 9. PROCESS ONLY TRUE NO-NAME MEDIA
# =========================
rows = []

for m in matches:
    grmd = m.group(1).strip()
    rest = m.group(2)

    # Only keep blocks that look like real media content
    if "\\mono{" not in rest and "\\chu{" not in rest:
        continue

    md_name_raw, body = split_name_and_body(rest)

    # STRICT: only true no-name media
    if not is_true_no_name(md_name_raw):
        continue

    # Combine \mono and \chu inner contents into ONE column
    combined_content = extract_all_blocks(body)

    # Skip empty combined content
    if not combined_content.strip():
        continue

    rows.append({
        "mediumID": grmd,
        "mediumName": f"JCM MEDIUM No. {grmd}",
        "ingredientName": combined_content
    })

# =========================
# 10. CONVERT TO DATAFRAME
# =========================
df = pd.DataFrame(rows)

# =========================
# 11. SAVE DIRECTLY TO FINAL CLEANED CSV
# =========================
df.to_csv(output_file, index=False, encoding="utf-8-sig")

# =========================
# 12. PRINT SUCCESS MESSAGE
# =========================
print(f"Done! Created {output_file} with {len(df)} cleaned no-name media rows.")

Found 1180 candidate blocks.
Done! Created jcm_cleaned_medium_data.csv with 34 cleaned no-name media rows.


In the code below i have to combined the three new csv files i have cleaned and prepared so i could do further analysis

In [ ]:
#setting file paths
file1 = "media_structured_JCM.csv"
file2 = "problematic_media_with_components.csv"
file3 = "jcm_cleaned_medium_data.csv"

output1 = "media_structured_JCM_modified.csv"
output2 = "problematic_media_with_components_modified.csv"
merged_output = "merged_jcm_files"

#we load our csv files
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)

#we will rename our columns
df1 = df1.rename(columns={"grmd":"mediumID",
                    "md_name":"mediumName",
                    "media_text":"ingredientName"})
df2 = df2.rename(columns={"grmd":"mediumID",
                    "md_name_original":"mediumName",
                    "components_text":"ingredientName"})
#in this code below we are saving the rename files
df1.to_csv(output1,index=False, encoding="utf-8")
df2.to_csv(output2, index=False, encoding="utf-8")
#confirmation
print("Renaming the file completed succesfully")
print(f"Saved renamed file1 as: {output1}")
print(f"Saved renamed file2 as: {output2}")


Renaming the file completed succesfully
Saved renamed file1 as: media_structured_JCM_modified.csv
Saved renamed file2 as: problematic_media_with_components_modified.csv


In this code i am trying to extract \chu text pattern as this will help me to clean my data properly

In this code below we are going to merged the new 3 csv files

In [ ]:
from IPython.utils import encoding
#here we define or setting the files path
file1 = "media_structured_JCM_modified.csv"
file2 = "problematic_media_with_components_modified.csv"
file3 = "jcm_cleaned_medium_data.csv"
#output file
merged_output = "merged_output_file.csv"
required_columns = ["mediumID","mediumName", "ingredientName"]
#the following codes helps us to load and validate
def load_and_validate(file_path,file_label):
  print(f"\nLoading {file_label}:{file_path}")
  df = pd.read_csv(file_path)
  print(f"Columns found in {file_label}:{df.columns.tolist()}")
  missing = [col for col in required_columns if col not in df.columns]
  if missing:
    raise ValueError(
      f"{file_label} is missing required columns: {missing}\n"
      f"Expected columns:{required_columns}\n"
      f"Found columns:{df.columns.to_list()}"
  )
#we are going to keep only required columns
  df = df[required_columns].copy()
  df["mediumID"] = df["mediumID"].astype(str).str.strip()
  df["mediumName"] = df["mediumName"].astype(str).str.strip()
  df["ingredientName"] = df["ingredientName"].astype(str).str.strip()

  return df

#load files
df1 = load_and_validate(file1,"File 1")
df2 = load_and_validate(file2, "File 2")
df3 = load_and_validate(file3, "File 3")
#concantenate the values
merged_df = pd.concat([df1,df2,df3], ignore_index = True)
print("\nMerge completed!")
print(f"Total rows BEFORE duplicate removal:{len(merged_df)}")
#removal of enpty rows
merged_df = merged_df [
     (merged_df["mediumID"]!= "") &
     (merged_df["mediumName"]!= "") &
     (merged_df["ingredientName"]!= "")
]
#removal of duplicates
merged_df = merged_df.drop_duplicates(subset=required_columns)
print(f"Total rows AFTER cleaning + duplicate removal:{len(merged_df)}")
#sorting of the values
merged_df = merged_df.sort_values(
    by=["mediumID","mediumName","ingredientName"]
).reset_index(drop=True)
#saving of the final file
merged_df.to_csv(merged_output, index=False, encoding="utf-8")

#final report
print(f"\nFinal merged file saved as: {merged_output}")
print("\nFinal columns:")
print(merged_df.columns.tolist())
print("\nfirst 10 rows of final merged file:")
print(merged_df.head(10))


Loading File 1:media_structured_JCM_modified.csv
Columns found in File 1:['mediumID', 'mediumName', 'ingredientName']

Loading File 2:problematic_media_with_components_modified.csv
Columns found in File 2:['mediumID', 'mediumName', 'md_name_cleaned', 'ingredientName', 'issue_types']

Loading File 3:jcm_cleaned_medium_data.csv
Columns found in File 3:['mediumID', 'mediumName', 'ingredientName']

Merge completed!
Total rows BEFORE duplicate removal:1136
Total rows AFTER cleaning + duplicate removal:1136

Final merged file saved as: merged_output_file.csv

Final columns:
['mediumID', 'mediumName', 'ingredientName']

first 10 rows of final merged file:
  mediumID                      mediumName  \
0        1                      MRS MEDIUM   
1      100          ALKALINE NUTRIENT AGAR   
2     1000                     PYVG MEDIUM   
3     1001       MARINE AGAR 2216 (pH 8.5)   
4     1002                     YTPS MEDIUM   
5     1003              ACIDIC HB-1 MEDIUM   
6     1004  FRESHWAT

 this code i am trying to extract \chu text pattern as this will help me to clean my data properly

In [ ]:
import pandas as pd
import re

# =========================
# 1. File paths
# =========================
input_file = "merged_output_file.csv"
output_file = "merged_output_with_method_formatted.csv"

# =========================
# 2. Load CSV
# =========================
df = pd.read_csv(input_file)

# =========================
# 3. Function to extract full \chu{...} blocks safely
#    (handles nested braces better than regex)
# =========================
def extract_chu_blocks(text):
    blocks = []
    i = 0

    while i < len(text):
        if text[i:i+5] == r'\chu{':
            start = i
            i += 5
            brace_count = 1

            while i < len(text) and brace_count > 0:
                if text[i] == '{':
                    brace_count += 1
                elif text[i] == '}':
                    brace_count -= 1
                i += 1

            block = text[start:i]
            blocks.append(block)
        else:
            i += 1

    return blocks

# =========================
# 4. Function to format ingredientName
#    Put each \mono and kept \chu on its own line
# =========================
def format_ingredient_lines(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Put a newline BEFORE each \mono or \chu if not already at start
    text = re.sub(r'(?<!^)(\\mono)', r'\n\1', text)
    text = re.sub(r'(?<!^)(\\chu)', r'\n\1', text)

    # Clean spaces around lines, keep line structure
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    return "\n".join(lines)

# =========================
# 5. Function to process ingredientName
# =========================
def process_ingredient(text):
    if pd.isna(text):
        return text, None

    text = str(text)
    chu_blocks = extract_chu_blocks(text)

    methods = []
    cleaned_text = text

    for block in chu_blocks:
        # Extract inner content safely
        if block.startswith(r'\chu{') and block.endswith('}'):
            inner_text = block[5:-1].strip()
        else:
            continue

        # Keep if pattern is \sfi something:
        if re.match(r'^\\sfi\s+.*:\s*$', inner_text):
            continue

        # Otherwise move to method
        methods.append(inner_text)
        cleaned_text = cleaned_text.replace(block, '', 1)

    # IMPORTANT: Do NOT collapse all whitespace into one line
    # Instead just tidy up small spaces around line breaks
    cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)      # normalize spaces only
    cleaned_text = re.sub(r' *\n *', '\n', cleaned_text)     # clean around existing newlines
    cleaned_text = cleaned_text.strip()

    # Reformat so each \mono and kept \chu starts on its own line
    cleaned_text = format_ingredient_lines(cleaned_text)

    # Join extracted methods
    method_text = " | ".join(methods) if methods else None

    return cleaned_text, method_text

# =========================
# 6. Apply to all rows
# =========================
processed = df["ingredientName"].apply(process_ingredient)

df["ingredientName"] = processed.apply(lambda x: x[0])   # cleaned + formatted ingredientName
df["method"] = processed.apply(lambda x: x[1])           # extracted method

# =========================
# 7. Keep required columns
# =========================
result = df[["mediumID", "mediumName", "ingredientName", "method"]]

# =========================
# 8. Save output
# =========================
result.to_csv(output_file, index=False)

print("Done! All rows processed successfully.")
print("Output saved as:", output_file)
print(result.head(20))

Done! All rows processed successfully.
Output saved as: merged_output_with_method_formatted.csv
    mediumID                         mediumName  \
0          1                         MRS MEDIUM   
1        100             ALKALINE NUTRIENT AGAR   
2       1000                        PYVG MEDIUM   
3       1001          MARINE AGAR 2216 (pH 8.5)   
4       1002                        YTPS MEDIUM   
5       1003                 ACIDIC HB-1 MEDIUM   
6       1004     FRESHWATER MEDIUM FOR SOIL AOA   
7       1005     HALOPHILE STARCH-CASEIN MEDIUM   
8       1007     MODIFIED HALOBACTERIA MEDIUM-2   
9       1008     MODIFIED HALOBACTERIA MEDIUM-3   
10      1009   CLOSTRIDIUM SWELLFUNIANUM MEDIUM   
11       101     BACILLUS ACIDOCALDARIUS MEDIUM   
12      1011                     SW--7.5 MEDIUM   
13      1012      ACIDITHRIX FERROOYDANS MEDIUM   
14      1013                 ACETIVIBRIO MEDIUM   
15      1014           NASU IRON REDUCER MEDIUM   
16      1015  MARINE AGAR 2216 WITH  

In the code below , i will clean the the csv file by removing latex and other complicated special characters

In [ ]:
import pandas as pd
import re

# =========================
# 1. File paths
# =========================
input_file = "merged_output_with_method_formatted.csv"
output_file = "merged_output_with_method_formatted_cleaned.csv"

# =========================
# 2. Load CSV
# =========================
df = pd.read_csv(input_file)

# =========================
# 3. Cleaning function
# =========================
def clean_ingredient(text):
    if pd.isna(text):
        return text

    text = str(text)

    # -------------------------
    # Step 1: Replace \hspace patterns
    # Handles both \hspace* and \hspace
    # -------------------------
    text = re.sub(
        r'\{([\d\.]+)\\hspace\*?\{\-\.5em\}\}',
        r'{\1-0.5}',
        text
    )

    # -------------------------
    # Step 2: Replace LaTeX symbols
    # -------------------------
    text = text.replace(r'\%', '%')
    text = text.replace(r'\cdot', '.')

    # -------------------------
    # Step 3: Remove $ and _
    # -------------------------
    text = text.replace('$', '')
    text = text.replace('_', '')

    # -------------------------
    # Step 4: Clean accidental double dots (optional safety)
    # -------------------------
    text = re.sub(r'\.\.+', '.', text)

    return text

# =========================
# 4. Apply cleaning
# =========================
df["ingredientName"] = df["ingredientName"].apply(clean_ingredient)

# =========================
# 5. Save output
# =========================
df.to_csv(output_file, index=False)

print("Cleaning complete!")
print("Saved as:", output_file)
print(df.head(20))

Cleaning complete!
Saved as: merged_output_with_method_formatted_cleaned.csv
    mediumID                         mediumName  \
0          1                         MRS MEDIUM   
1        100             ALKALINE NUTRIENT AGAR   
2       1000                        PYVG MEDIUM   
3       1001          MARINE AGAR 2216 (pH 8.5)   
4       1002                        YTPS MEDIUM   
5       1003                 ACIDIC HB-1 MEDIUM   
6       1004     FRESHWATER MEDIUM FOR SOIL AOA   
7       1005     HALOPHILE STARCH-CASEIN MEDIUM   
8       1007     MODIFIED HALOBACTERIA MEDIUM-2   
9       1008     MODIFIED HALOBACTERIA MEDIUM-3   
10      1009   CLOSTRIDIUM SWELLFUNIANUM MEDIUM   
11       101     BACILLUS ACIDOCALDARIUS MEDIUM   
12      1011                     SW--7.5 MEDIUM   
13      1012      ACIDITHRIX FERROOYDANS MEDIUM   
14      1013                 ACETIVIBRIO MEDIUM   
15      1014           NASU IRON REDUCER MEDIUM   
16      1015  MARINE AGAR 2216 WITH  8.0\% NaCl   
17   

Removal of -- to - and removal of \hspace{-{}} to {} only


In [ ]:
import pandas as pd
import re

# =========================
# 1. Load CSV (FIXED)
# =========================
df = pd.read_csv("merged_output_with_method_formatted_cleaned.csv", engine="python")

# Clean column names
df.columns = df.columns.str.strip()

# =========================
# 2. Format lines function
# =========================
def format_lines(text):
    if pd.isna(text):
        return text

    text = str(text)

    text = re.sub(r'(?<!^)(\\mono)', r'\n\1', text)
    text = re.sub(r'(?<!^)(\\chu)', r'\n\1', text)

    lines = [line.strip() for line in text.splitlines() if line.strip()]

    return "\n".join(lines)

# =========================
# 3. Cleaning function
# =========================
def clean_ingredient(text):
    if pd.isna(text):
        return text

    text = str(text)

    text = text.replace("--", "-")
    text = text.replace("*", "")

    text = re.sub(
        r'\{([\d\.]+)\\hspace\*?\{\-0\.5em\}\}',
        r'{\1-0.5}',
        text
    )

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)

    text = text.strip() if text else text

    text = format_lines(text)

    return text

# =========================
# 4. Apply cleaning
# =========================
df["ingredientName"] = df["ingredientName"].apply(clean_ingredient)

# =========================
# 5. Save
# =========================
df.to_csv("merged_output_cleaned_ingredients.csv", index=False)

print("Done!")
print(df[["ingredientName"]].head(10))

Done!
                                      ingredientName
0  \mono{Casein peptone, tryptic digest}{10.0}{g}...
1  \mono{Peptone}{5.0}{g}\n\mono{Beef extract}{3....
2  \mono{Trypticase peptone (BD-BBL)}{5.0}{g}\n\m...
3                                                NaN
4  \mono{Yeast extract (BD-Difco)}{5.0}{g}\n\mono...
5  \mono{10% KNO3 solution}{20.0}{ml}\n\mono{20% ...
6  \chu{\sfi Basic FWM solution:}\n\mono{NaCl}{1....
7  \mono{Soluble starch}{100.0}{g}\n\mono{Casein}...
8                                                NaN
9                                                NaN


In [ ]:
import pandas as pd
import re

# =========================
# 1. File paths
# =========================
input_file = "merged_output_cleaned_ingredients.csv"
output_file = "mono_transformed.csv"

# =========================
# 2. Load CSV (safe mode)
# =========================
df = pd.read_csv(input_file, engine="python")
df.columns = df.columns.str.strip()

# =========================
# 3. Function to transform \mono{}
# =========================
def transform_mono(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Pattern to capture:
    # \mono{compound}{value}{unit}
    pattern = r'\\mono\{(.*?)\}\{(.*?)\}\{(.*?)\}'

    # Replace with:
    # compound (value) (unit)
    def replace_func(match):
        compound = match.group(1).strip()
        value = match.group(2).strip()
        unit = match.group(3).strip()

        return f"{compound} ({value}) ({unit})"

    # Replace ALL occurrences in the text
    text = re.sub(pattern, replace_func, text)

    return text

# =========================
# 4. Apply transformation
# =========================
df["ingredientName"] = df["ingredientName"].apply(transform_mono)

# =========================
# 5. Save output
# =========================
df.to_csv(output_file, index=False)

print("Transformation complete!")
print("Saved as:", output_file)
print(df.head(10))


Transformation complete!
Saved as: mono_transformed.csv
   mediumID                      mediumName  \
0         1                      MRS MEDIUM   
1       100          ALKALINE NUTRIENT AGAR   
2      1000                     PYVG MEDIUM   
3      1001       MARINE AGAR 2216 (pH 8.5)   
4      1002                     YTPS MEDIUM   
5      1003              ACIDIC HB-1 MEDIUM   
6      1004  FRESHWATER MEDIUM FOR SOIL AOA   
7      1005  HALOPHILE STARCH-CASEIN MEDIUM   
8      1007  MODIFIED HALOBACTERIA MEDIUM-2   
9      1008  MODIFIED HALOBACTERIA MEDIUM-3   

                                      ingredientName  \
0  Casein peptone, tryptic digest (10.0) (g)\nBee...   
1  Peptone (5.0) (g)\nBeef extract (3.0) (g)\nAga...   
2  Trypticase peptone (BD-BBL) (5.0) (g)\nBacto p...   
3                                                NaN   
4  Yeast extract (BD-Difco) (5.0) (g)\nTryptone (...   
5  10% KNO3 solution (20.0) (ml)\n20% MES (pH 5.5...   
6  \chu{\sfi Basic FWM solution:}\

In the code below we are going to do some count of how many ingredients are in each rows etc

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving mono_transformed_new.csv to mono_transformed_new (1).csv


In [ ]:
with open("mono_transformed_new.csv", "r") as f:
    print(f.readline())

mediumID,mediumName,ingredientName,method



In [ ]:
import pandas as pd
import re
from collections import Counter

# =========================
# 1. File paths
# =========================
input_file = "mono_transformed_new.csv"

output_medium_counts = "medium_ingredient_counts.csv"
output_ingredient_freq = "ingredient_frequency.csv"

# =========================
# 2. Load CSV
# =========================
df = pd.read_csv(input_file, engine="python")
df.columns = df.columns.str.strip()

# =========================
# 3. Function to extract valid ingredient lines
# =========================
def extract_ingredients(text):
    if pd.isna(text):
        return []

    text = str(text)

    # Split into lines
    lines = text.split("\n")

    valid_ingredients = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # ❌ Skip unwanted lines
        if line.startswith(r"\chu"):
            continue
        if line.startswith(r"\sfi"):
            continue

        # ✅ Only keep lines with (value) (unit)
        if re.search(r'\(.+?\)\s*\(.+?\)', line):
            valid_ingredients.append(line)

    return valid_ingredients

# =========================
# 4. TASK 1: Count ingredients per medium
# =========================
medium_counts = []

# =========================
# 5. TASK 2: Count ingredient frequency
# =========================
ingredient_counter = Counter()

for _, row in df.iterrows():
    medium_id = row["mediumID"]
    medium_name = row["mediumName"]
    text = row["ingredientName"]

    ingredients = extract_ingredients(text)

    # Count per medium
    medium_counts.append({
        "mediumID": medium_id,
        "mediumName": medium_name,
        "ingredient_count": len(ingredients)
    })

    # Count global ingredient frequency
    for ing in ingredients:
        # Extract ONLY ingredient name (before first "(")
        name = ing.split("(")[0].strip()
        ingredient_counter[name] += 1

# =========================
# 6. Save TASK 1 output
# =========================
df_medium_counts = pd.DataFrame(medium_counts)
df_medium_counts.to_csv(output_medium_counts, index=False)

# =========================
# 7. Save TASK 2 output
# =========================
df_ingredient_freq = pd.DataFrame(
    ingredient_counter.items(),
    columns=["ingredient", "count"]
).sort_values(by="count", ascending=False)

df_ingredient_freq.to_csv(output_ingredient_freq, index=False)

# =========================
# 8. Done
# =========================
print("Done!")

print("\nTop 10 most common ingredients:")
print(df_ingredient_freq.head(10))

print("\nSample medium counts:")
print(df_medium_counts.head(10))

Done!

Top 10 most common ingredients:
         ingredient  count
10  Distilled water   1339
24             NaCl    669
2     Yeast extract    601
21       CaCl2.2H2O    505
22           KH2PO4    454
71            NH4Cl    402
8        MgSO4.7H2O    394
28              KCl    378
27       MgCl2.6H2O    371
36        Resazurin    337

Sample medium counts:
   mediumID                      mediumName  ingredient_count
0         1                      MRS MEDIUM                11
1       100          ALKALINE NUTRIENT AGAR                 4
2      1000                     PYVG MEDIUM                18
3      1001       MARINE AGAR 2216 (pH 8.5)                 0
4      1002                     YTPS MEDIUM                18
5      1003              ACIDIC HB-1 MEDIUM                 3
6      1004  FRESHWATER MEDIUM FOR SOIL AOA                35
7      1005  HALOPHILE STARCH-CASEIN MEDIUM                 7
8      1007  MODIFIED HALOBACTERIA MEDIUM-2                 0
9      1008  MODIFIED